In [61]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [4]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [19]:
data = torch.arange(1, 18, dtype=torch.float32).view(-1, 1)  # [1..10]
X = []
y = []
seq_len = 3
for i in range(len(data) - seq_len):
    X.append(data[i:i+seq_len])
    y.append(data[i+seq_len])
X = torch.stack(X)
y = torch.stack(y)

# 🔹 Przenosimy dane na urządzenie
X, y = X.to(device), y.to(device)

# 🔹 Definicja modelu LSTM
class SimpleLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, output_size=1):
        super(SimpleLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # ostatni krok czasowy
        return out

# 🔹 Inicjalizacja modelu
model = SimpleLSTM().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 🔹 Trening
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.5f}")

# 🔹 Predykcja
model.eval()
test_seq = torch.tensor([[8.0], [9.0], [10.0]]).unsqueeze(0).to(device)
pred = model(test_seq)
print(f"Predykcja następnej liczby po [8,9,10]: {pred.item():.2f}")

Epoch 50, Loss: 23.58110
Epoch 100, Loss: 5.79698
Epoch 150, Loss: 0.89161
Epoch 200, Loss: 0.13620
Epoch 250, Loss: 0.02401
Epoch 300, Loss: 0.00901
Predykcja następnej liczby po [8,9,10]: 11.00


In [43]:
# prepare data

# h1 close diff
# m1 close diff
# h4 close diff

periods_map = {
    1: 1,
    7: 15,
    8: 30,
    9: 60,
    10: 240
}

def load_csv(path, shift_to_close=True):
    df = pd.read_csv(path)
    df["open"]  = df["low"] + df["delta_open"]
    df["close"] = df["low"] + df["delta_close"]
    df["high"]  = df["low"] + df["delta_high"]
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='raise')

    period = int(df['period'].iloc[0]) if 'period' in df.columns else None
    period_min = periods_map.get(period)
    if shift_to_close and period_min is not None:
        # ustawiamy datetime = czas zamknięcia świecy = start + period
        df['datetime'] = df['timestamp'] + pd.to_timedelta(period_min, unit='m')
    else:
        # traktujemy timestamp jako już poprawny (np. close time)
        df['datetime'] = df['timestamp']

    # wybieramy kolumny interesujące
    df = df[["datetime", "open", "high", "low", "close", "volume"]].sort_values("datetime")
    return df.reset_index(drop=True)

    # df['datetime'] = pd.to_datetime(df['timestamp'])
    # df = df.set_index('datetime').sort_index()
    # df = df[["open", "high", "low", "close", "volume"]]
    # return df

m1_df = load_csv("data/XTIUSD_20251018_2200_20251025_2159_M1.csv")
m15_df = load_csv("data/XTIUSD_20251018_2200_20251025_2159_M15.csv")
h1_df = load_csv("data/XTIUSD_20251018_2200_20251025_2159_H1.csv")
h4_df = load_csv("data/XTIUSD_20251018_2200_20251025_2159_H4.csv")

m1_df  = m1_df.sort_values("datetime")
m15_df = m15_df.sort_values("datetime")
h1_df  = h1_df.sort_values("datetime")
h4_df  = h4_df.sort_values("datetime")

# Pomocniczna funkcja do merge_asof i dopisania sufiksu
def asof_merge(base_df, other_df, suffix):
    # merge_asof łączy każdą datę base_df z ostatnią datą <= tej daty w other_df
    merged = pd.merge_asof(
        base_df,
        other_df,
        on="datetime",
        direction="backward",
        suffixes=("", f"_{suffix}")
    )
    # jeżeli kolumny bez sufiksu występują w obu df, pandas doda suffix do drugiego.
    # Upewnijmy się, że nazwy są spójne: jeśli powstały kolumny typu 'open_m15' lub 'open_m15' - ok.
    return merged

# df = m1_df.copy()
df = m15_df.copy()
# df = asof_merge(df, m15_df, "m15")
df = asof_merge(df, h1_df,  "h1")
df = asof_merge(df, h4_df,  "h4")

cols = ["datetime",
        "open","high","low","close","volume",
        "open_m15","high_m15","low_m15","close_m15","volume_m15",
        "open_h1","high_h1","low_h1","close_h1","volume_h1",
        "open_h4","high_h4","low_h4","close_h4","volume_h4"]
# zachowaj tylko istniejące kolumny (dla bezpieczeństwa)
cols = [c for c in cols if c in df.columns]
df = df[cols]

# ustaw index na datetime (opcjonalnie)
df = df.set_index("datetime")

# Zapisz wynik
df.to_csv("data/XTIUSD_merged_M1_with_higher_timeframes.csv", index=True)

# gotowe
print("Scalono pliki. Wynik zapisany jako: data/XTIUSD_merged_M1_with_higher_timeframes.csv")


Scalono pliki. Wynik zapisany jako: data/XTIUSD_merged_M1_with_higher_timeframes.csv


In [44]:
df.columns

Index(['open', 'high', 'low', 'close', 'volume', 'open_h1', 'high_h1',
       'low_h1', 'close_h1', 'volume_h1', 'open_h4', 'high_h4', 'low_h4',
       'close_h4', 'volume_h4'],
      dtype='object')

In [67]:
new_df = df[['close', 'close_h1', 'close_h4', 'high_h1', 'low_h1']].dropna()
new_df['close_diff'] = new_df['close'].diff()
new_df['min_max_h1'] = new_df['high_h1'] - new_df['low_h1']
new_df = new_df.drop(columns=['low_h1', 'high_h1'])
new_df['target_close'] = new_df['close'].shift(-1)
new_df = new_df.dropna()
new_df = new_df.loc[(new_df.index.hour >= 8) & (new_df.index.hour <= 19)]
new_df

,close,close_h1,close_h4,close_diff,min_max_h1,target_close
datetime,,,,,,
2025-10-20 08:00:00,57.07,57.07,57.39,-0.12,0.31,57.09
2025-10-20 08:15:00,57.09,57.07,57.39,0.02,0.31,57.19
2025-10-20 08:30:00,57.19,57.07,57.39,0.10,0.31,57.27
2025-10-20 08:45:00,57.27,57.07,57.39,0.08,0.31,57.35
2025-10-20 09:00:00,57.35,57.35,57.35,0.08,0.32,57.35
...,...,...,...,...,...,...
2025-10-24 18:45:00,61.95,61.89,62.26,0.13,0.43,61.91
2025-10-24 19:00:00,61.91,61.91,62.26,-0.04,0.27,61.87
2025-10-24 19:15:00,61.87,61.91,62.26,-0.04,0.27,61.81


In [68]:
feature_cols = ['close', 'close_h1', 'close_h4', 'close_diff', 'min_max_h1']
X = new_df[feature_cols].values.astype(np.float32)
y = new_df['target_close'].values.astype(np.float32).reshape(-1, 1)
